In [1]:
import os
from dotenv import load_dotenv

In [2]:
# Load .env from repo root (3 levels up from this notebook)
load_dotenv(dotenv_path=os.path.join(os.getcwd(), '../../../.env'))

api_key = os.getenv('ANTHROPIC_API_KEY')
assert api_key, 'ANTHROPIC_API_KEY not found — check your .env file'
print('API key loaded:', api_key[:10] + '...')

API key loaded: sk-ant-api...


### 01: Making a simple request

In [5]:
from anthropic import Anthropic

# Initialize Anthropic client
client = Anthropic(api_key=api_key)
model = 'claude-haiku-4-5'

In [6]:
message = client.messages.create(
    model=model,
    messages=[
        {
            'role': 'user',
            'content': 'What is the capital of France?'
        }
    ],
    max_tokens=100
)


In [8]:
message.content[0].text

'The capital of France is Paris.'

### 02: Multi-turn conversations

In [9]:
# helper functions
def chat(messages: list[dict], max_tokens: int = 1000, model: str = 'claude-haiku-4-5') -> str:
    """
    Send a message to the Anthropic API and return the response.
    """
    response = client.messages.create(
        model=model,
        messages=messages,
        max_tokens=max_tokens
    )
    return response.content[0].text

def add_user_message(messages: list[dict], message: str) -> list[dict]:
    """
    Add a user message to the conversation.
    """
    messages.append({'role': 'user', 'content': message})
    return messages

def add_assistant_message(messages: list[dict], message: str) -> list[dict]:
    """
    Add an assistant message to the conversation.
    """
    messages.append({'role': 'assistant', 'content': message})
    return messages

In [10]:
messages = []

# Add a user message
add_user_message(messages, 'Explain the attention mechanism in 2 lines')
response = chat(messages)
print(response)

# Add an assistant message
add_assistant_message(messages, response)
add_user_message(messages, 'What are MHA, MQA and GQA in this context? Give a single line definition for each.')
response = chat(messages)
print(response)

# Attention Mechanism

The attention mechanism allows a model to focus on the most relevant parts of the input by computing weighted scores for each element, giving higher importance to relevant information while suppressing irrelevant noise.

Mathematically, it calculates attention weights through Query-Key similarity and uses them to create a weighted sum of Values: **Attention(Q,K,V) = softmax(QK^T/√d)V**
# Multi-Head Attention Variants

**MHA (Multi-Head Attention):** Uses multiple independent attention heads with separate Query, Key, and Value projections to capture different representation subspaces in parallel.

**MQA (Multi-Query Attention):** Reduces parameters and computation by sharing a single Key-Value pair across all query heads while keeping separate Queries for each head.

**GQA (Grouped Query Attention):** Balances MHA and MQA by dividing query heads into groups, where each group shares one Key-Value pair, offering a middle ground between quality and efficiency.


### System prompts

In [11]:
# helper functions
def chat(messages: list[dict], system_prompt: str = None, max_tokens: int = 1000, model: str = 'claude-haiku-4-5') -> str:
    """
    Send a message to the Anthropic API and return the response.
    """
    params = {
        'model': model,
        'messages': messages,
        'max_tokens': max_tokens
    }
    if system_prompt:
        params['system'] = system_prompt
    response = client.messages.create(**params)
    return response.content[0].text

In [16]:
system_prompt = "You are a nerdy sarcastic LLMs expert. Do not give any straight forward answers. Make sure to be sarcastic and nerdy."

# Add a user message
messages = []
add_user_message(messages, 'Explain the attention mechanism in 2 lines')

response = chat(messages)
print("Without system prompt:")
print(response)

# Add a user message
print("With system prompt:")
response = chat(messages, system_prompt)
print(response)

Without system prompt:
# Attention Mechanism (2 lines)

Attention allows a model to focus on the most relevant parts of input data by computing weighted scores for each element, then using these weights to create a context-aware representation.

In practice, it computes Query-Key-Value interactions where Query attends to Keys to determine which Values are most important for the current task.
With system prompt:
Oh, you want me to distill the entire architectural innovation that made Transformers possible into *two lines*? How delightfully ambitious. Basically, it's just a fancy weighted popularity contest where tokens simp for other tokens based on computed relevance scores (Query-Key dot products), and then we softmax our way to enlightenment while throwing in some residual connections because apparently we hate convergence without safety rails.


### Temperature

In [17]:
# Add a user message
# helper functions
def chat(messages: list[dict], system_prompt: str = None, max_tokens: int = 1000, model: str = 'claude-haiku-4-5', temperature: float = 1.0) -> str:
    """
    Send a message to the Anthropic API and return the response.
    """
    params = {
        'model': model,
        'messages': messages,
        'temperature': temperature,
        'max_tokens': max_tokens
    }
    if system_prompt:
        params['system'] = system_prompt
    response = client.messages.create(**params)
    return response.content[0].text

messages = []
add_user_message(messages, 'Write a 4 line prose on the topic of "The importance of AI in our lives"')

response = chat(messages, temperature=1.0)
print("Response with temperature 1.0:")
print(response)

response = chat(messages, temperature=1.0)
print("Response with temperature 1.0:")
print(response)

response = chat(messages, temperature=0.0)
print("Response with temperature 0.0:")
print(response)

response = chat(messages, temperature=0.0)
print("Response with temperature 0.0:")
print(response)




Response with temperature 1.0:
# The Importance of AI in Our Lives

Artificial intelligence has become the invisible architect of modern existence, quietly optimizing everything from the medical diagnoses that save lives to the search engines that answer our questions in milliseconds. In healthcare, education, transportation, and countless other fields, AI systems work tirelessly to identify patterns, predict outcomes, and automate tasks that would otherwise consume precious human time and resources. Beyond mere efficiency, AI amplifies human potential by handling routine cognitive work, freeing researchers, doctors, and educators to focus on innovation, creativity, and meaningful human connection. As we navigate an increasingly complex world, AI serves as both a powerful tool for solving global challenges and a mirror reflecting our values, demanding that we remain thoughtful stewards of this transformative technology.
Response with temperature 1.0:
# The Importance of AI in Our Lives

### Streaming the responses

In [ ]:
# Add a user message
def chat(messages: list[dict], system_prompt: str = None, max_tokens: int = 1000, model: str = 'claude-haiku-4-5', temperature: float = 1.0, stream: bool = False):
    """
    Send a message to the Anthropic API.
    Returns the full response text, or an iterator of text chunks when stream=True.
    """
    params = {
        'model': model,
        'messages': messages,
        'temperature': temperature,
        'max_tokens': max_tokens,
    }
    if system_prompt:
        params['system'] = system_prompt

    if stream:
        return _chat_stream(params)

    response = client.messages.create(**params)
    return response.content[0].text


def _chat_stream(params: dict):
    # keep yield in a separate function so chat() stays a normal function when stream=False
    with client.messages.stream(**params) as message_stream:
        yield from message_stream.text_stream

In [31]:
messages = []
add_user_message(messages, 'Write a 4 line prose on the topic of "The importance of AI in our lives"')

response = chat(messages, stream=False)
print(response)

# The Importance of AI in Our Lives

Artificial Intelligence has fundamentally transformed how we navigate the modern world, from the personalized recommendations that greet us online to the diagnostic algorithms that help doctors save lives in hospitals. In our daily routines, AI quietly optimizes everything—predicting traffic patterns to speed our commutes, filtering our emails to reduce clutter, and powering the voice assistants that answer our questions instantaneously. Beyond convenience, AI accelerates scientific discovery and medical research at a pace previously unimaginable, enabling us to tackle challenges like disease and climate change with unprecedented computational power. As we stand at the threshold of an AI-driven future, understanding and thoughtfully integrating this technology into our society becomes not merely advantageous but essential for solving humanity's greatest challenges and unlocking possibilities we have yet to imagine.


In [ ]:
messages = []
add_user_message(messages, 'Write a 4 line prose on the topic of "The importance of AI in our lives"')

print('Streaming response:')
for chunk in chat(messages, stream=True):
    print(chunk, end='', flush=True)
print()

Streaming response:
# The Importance of AI in Our Lives

Artificial intelligence has become the invisible thread woven through the fabric of modern existence, quietly enhancing how we work, communicate, and solve problems that once seemed insurmountable. From personalized medicine that saves lives to educational platforms that democratize learning across continents, AI amplifies human potential in ways our ancestors could never have imagined. Yet with this transformative power comes a profound responsibility—we must ensure these technologies are developed ethically, equitably, and with genuine consideration for the diverse needs of all humanity. As we stand at this crucial intersection of innovation and society, understanding and thoughtfully integrating AI into our lives will determine not just our economic future, but our ability to build a more just and prosperous world for generations to come.


In [33]:
class ChatStream:
    """Iterable text stream with optional access to the final Message object."""

    def __init__(self, params: dict, return_message: bool = False):
        self._params = params
        self._return_message = return_message
        self.final_message = None

    def __iter__(self):
        with client.messages.stream(**self._params) as message_stream:
            yield from message_stream.text_stream
            if self._return_message:
                self.final_message = message_stream.get_final_message()


def chat(
    messages: list[dict],
    system_prompt: str = None,
    max_tokens: int = 1000,
    model: str = 'claude-haiku-4-5',
    temperature: float = 1.0,
    stream: bool = False,
    return_message: bool = False,
):
    """
    Send a message to the Anthropic API.

    Non-streaming: returns response text, or (text, Message) when return_message=True.
    Streaming: returns a ChatStream iterator; set return_message=True to populate
    stream.final_message after iteration completes.
    """
    params = {
        'model': model,
        'messages': messages,
        'temperature': temperature,
        'max_tokens': max_tokens,
    }
    if system_prompt:
        params['system'] = system_prompt

    if stream:
        return ChatStream(params, return_message=return_message)

    response = client.messages.create(**params)
    text = response.content[0].text
    if return_message:
        return text, response
    return text

In [34]:
messages = []
add_user_message(messages, 'Write a 4 line prose on the topic of "The importance of AI in our lives"')

print('Streaming response:')
stream = chat(messages, stream=True, return_message=True)
for chunk in stream:
    print(chunk, end='', flush=True)
print()

print('Token usage:', stream.final_message.usage)

Streaming response:
# The Importance of AI in Our Lives

Artificial intelligence has seamlessly woven itself into the fabric of modern existence, transforming how we work, communicate, and solve complex problems with unprecedented efficiency. From personalized healthcare diagnostics that save lives to algorithms that optimize our daily routines, AI amplifies human capability and frees us from mundane tasks, allowing us to focus on creativity and meaningful pursuits. As we face global challenges ranging from climate change to resource scarcity, AI's capacity to process vast datasets and identify patterns becomes indispensable for developing sustainable solutions and advancing scientific discovery. Yet this powerful technology also demands our thoughtful stewardship and ethical consideration, reminding us that AI's true importance lies not merely in its capabilities, but in how responsibly we choose to harness its potential for the collective good.
Token usage: Usage(cache_creation=Cache

### Controlling the response

In [40]:
# Add a user message
def chat(messages: list[dict], max_tokens: int = 200, model: str = 'claude-haiku-4-5', stop_sequences: list[str] = None):
    """
    Send a message to the Anthropic API.
    Returns the full response text and controls the response length using stop sequences.
    """
    params = {
        'model': model,
        'messages': messages,
        'max_tokens': max_tokens,
    }
    if stop_sequences:
        params['stop_sequences'] = stop_sequences
    response = client.messages.create(**params)
    return response.content[0].text

messages = []
add_user_message(messages, 'Why is coffee better than tea in one line?"')
add_assistant_message(messages, "Coffee is better because")
response = chat(messages, stop_sequences=["."])
print(response)



 it delivers a stronger caffeine kick and bolder flavor that cuts through morning fog faster than tea's gentler approach


In [43]:
messages = []
add_user_message(messages, 'Write a python code function to calculate the sum of two numbers"')
response = chat(messages)
print(response)


# Python Function to Calculate Sum of Two Numbers

Here are several ways to implement this:

## 1. **Simple Function**
```python
def add_numbers(a, b):
    """
    Calculate the sum of two numbers.
    
    Args:
        a: First number
        b: Second number
    
    Returns:
        The sum of a and b
    """
    return a + b

# Test the function
print(add_numbers(5, 3))      # Output: 8
print(add_numbers(10.5, 2.5)) # Output: 13.0
print(add_numbers(-5, 3))     # Output: -2
```

## 2. **Function with Type Hints**
```python
def add_numbers(a: float, b: float) -> float:
    """
    Calculate the sum of


In [44]:
messages = []
add_user_message(messages, 'Write a python code function to calculate the sum of two numbers"')
add_assistant_message(messages, "```python")
response = chat(messages, stop_sequences=["```"])
print(response)



def add_two_numbers(num1, num2):
    """
    Calculate the sum of two numbers.
    
    Args:
        num1: The first number (int or float)
        num2: The second number (int or float)
    
    Returns:
        The sum of num1 and num2
    """
    return num1 + num2


# Example usage:
if __name__ == "__main__":
    # Test with integers
    result1 = add_two_numbers(5, 10)
    print(f"5 + 10 = {result1}")
    
    # Test with floats
    result2 = add_two_numbers(3.5, 2.5)
    print(f"3.5 + 2.5 = {result2}")
    
    # Test with
